# Stage 5 — Embeddings

**Project:** ResearchMate — Research Paper RAG Chatbot
**Goal of this notebook:** Load a local embedding model, understand what an embedding vector looks like, verify that similar texts get higher similarity scores than unrelated ones, and test embedding a small sample of our real documents.

**Before running:** upload `train_prepared.csv` (from Stage 2) to this Colab session.

## Cell 1 — Install packages

`langchain-huggingface` gives us the LangChain wrapper around HuggingFace models. `sentence-transformers` is the underlying library that actually runs the embedding model locally. We also reuse `langchain-core` and `langchain-text-splitters` from earlier stages to rebuild our document list.

In [1]:
!pip install -q langchain-huggingface sentence-transformers langchain-core langchain-text-splitters

## Cell 2 — Reload data and rebuild documents (same as Stages 3-4)

Self-contained setup: load the prepared CSV, rebuild the `Document` list, and split (Stage 4 already confirmed this leaves every document as a single chunk).

In [2]:
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

df = pd.read_csv("train_prepared.csv")

documents = []
for _, row in df.iterrows():
    doc = Document(
        page_content=row["text"],
        metadata={
            "id": row["ID"],
            "title": row["TITLE"],
            "topics": row["topics"],
        }
    )
    documents.append(doc)

splitter = RecursiveCharacterTextSplitter(chunk_size=3500, chunk_overlap=200)
split_docs = splitter.split_documents(documents)

print("Total chunks ready for embedding:", len(split_docs))

Total chunks ready for embedding: 20971


## Cell 3 — Load the embedding model

`sentence-transformers/all-MiniLM-L6-v2`, run locally — no API key needed. The first run downloads the model (~80MB) into the Colab session.

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


## Cell 4 — Embed a single piece of text and inspect the vector

We embed one plain sentence and look at the result: how many numbers it has (the vector's dimension), and what the first few values look like. This makes "embedding" concrete instead of abstract.

In [4]:
sample_text = "Graph neural networks for node classification tasks."

vector = embedding_model.embed_query(sample_text)

print("Vector length (dimensions):", len(vector))
print("First 10 values:", vector[:10])

Vector length (dimensions): 384
First 10 values: [-0.09116603434085846, -0.0486631877720356, 0.05045975744724274, 0.006074532866477966, 0.02623787894845009, 0.034249503165483475, -0.019830960780382156, -0.07308705896139145, -0.106361523270607, 0.016102783381938934]


## Cell 5 — Verify similarity behaves as expected

We embed 3 short texts: two about related topics (graph neural networks / GNNs for node classification), and one about something unrelated (cooking a recipe). We then compute cosine similarity between each pair using plain NumPy. If embeddings work as intended, the two related texts should have a noticeably higher similarity score than either has with the unrelated one.

In [5]:
import numpy as np

text_a = "Graph neural networks for node classification tasks."
text_b = "GNN-based approaches to classify nodes in a graph."
text_c = "A simple recipe for baking chocolate chip cookies."

vec_a = np.array(embedding_model.embed_query(text_a))
vec_b = np.array(embedding_model.embed_query(text_b))
vec_c = np.array(embedding_model.embed_query(text_c))

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

print("Similarity (A vs B, related topics):   ", cosine_similarity(vec_a, vec_b))
print("Similarity (A vs C, unrelated topics): ", cosine_similarity(vec_a, vec_c))
print("Similarity (B vs C, unrelated topics): ", cosine_similarity(vec_b, vec_c))

Similarity (A vs B, related topics):    0.7562439471705046
Similarity (A vs C, unrelated topics):  0.0533518750608722
Similarity (B vs C, unrelated topics):  0.04681588390370389


## Cell 6 — Test embedding a small sample of our real documents

Before embedding all 20,971 chunks (which we'll do as part of building the vector store in Stage 6), we test the process on just 5 real documents — to confirm it works correctly and to get a sense of timing.

In [6]:
import time

sample_docs = split_docs[:5]
sample_texts = [doc.page_content for doc in sample_docs]

start = time.time()
sample_vectors = embedding_model.embed_documents(sample_texts)
elapsed = time.time() - start

print(f"Embedded {len(sample_vectors)} documents in {elapsed:.2f} seconds")
print("Each vector has", len(sample_vectors[0]), "dimensions")
print(f"\nEstimated time for all {len(split_docs)} chunks: {elapsed / len(sample_docs) * len(split_docs):.1f} seconds")

Embedded 5 documents in 2.70 seconds
Each vector has 384 dimensions

Estimated time for all 20971 chunks: 11315.0 seconds


## What to check after running this notebook

- **Cell 3:** model loads without errors (first run may take a minute to download).
- **Cell 4:** vector length should be **384** — confirms we're using the model we expect.
- **Cell 5:** the related pair (A vs B) should have a clearly higher similarity score than either related-vs-unrelated pair (A vs C, B vs C). This is the core proof that the embedding model actually understands meaning, not just word overlap.
- **Cell 6:** 5 documents embed successfully, each producing a 384-dimensional vector. Note the estimated time for the full corpus — this tells us what to expect when we build the vector store in Stage 6.

Paste back the vector dimension (Cell 4), the three similarity scores (Cell 5), and the estimated full-corpus time (Cell 6) — then we'll move to Stage 6 (Vector Store), where we actually embed and store all 20,971 chunks.